# Generar el corpus sintètic valencià → català con qwen2.5:14b en Colab

Este notebook ejecuta `sinteticos/genera_corpus_sintetico.py` con GPU en
Google Colab, para probar si compensa frente a correrlo en CPU local o
esperar al clúster SLURM de la universidad.

Usa el **mismo motor** que ya validaste en local: el system prompt y el
glosario dinámico de `03_seleccion_de_modelo/evalua_models.py`, con el modelo
qwen2.5:14b que dio BLEU 84,52 / chrF 92,34 en el benchmark.

## Antes de empezar

Colab no ve tu disco local, así que necesitas subir el proyecto a **Google
Drive** primero. Sube (o sincroniza) al menos estas carpetas/archivos,
manteniendo la misma estructura relativa:

```
MyDrive/AVLizador/scrapeo/
├── 03_seleccion_de_modelo/
│   ├── evalua_models.py
│   └── palabras_traducidas.json
├── corpus/final/dialectal/
│   └── corpus_occidental_net.jsonl
└── sinteticos/
    └── genera_corpus_sintetico.py
```

(Lo más simple: comprime toda la carpeta `scrapeo` en un .zip y súbelo a tu
Drive, descomprimido en `MyDrive/AVLizador/scrapeo/`. No hace falta que subas
`corpus/raw`, `corpus/clean`, `03_seleccion_de_modelo/resultats`, etc. — solo lo de
la lista de arriba hace falta para que el script funcione.)

## Activa la GPU

Antes de ejecutar nada: `Entorno de ejecución` → `Cambiar tipo de entorno de
ejecución` → GPU (T4 es la que suele tocar gratis).


In [4]:
# Comprueba que Colab te ha dado una GPU
!nvidia-smi


Wed Sep  9 08:17:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Montar Google Drive

Te pedirá autorizar el acceso — es necesario para leer el proyecto y para
que el corpus generado y el modelo de Ollama sobrevivan aunque se corte la
sesión de Colab (el disco local de la VM se borra en cada sesión nueva).


In [8]:
from google.colab import drive
# drive.mount('/content/drive', force_remount=True)
drive.mount('/content/drive')


Mounted at /content/drive


## 2. Configura la ruta a tu proyecto

Cambia `PROJECT_DIR` si subiste la carpeta a otro sitio de tu Drive.


In [9]:
import os

PROJECT_DIR = "/content/drive/MyDrive/AVLizador/scrapeo"  # <-- CAMBIA ESTO si tu ruta es distinta
SINTETICOS_DIR = os.path.join(PROJECT_DIR, "sinteticos")

# Comprueba que están los archivos imprescindibles antes de seguir
requeridos = [
    "03_seleccion_de_modelo/evalua_models.py",
    "03_seleccion_de_modelo/palabras_traducidas.json",
    "corpus/final/dialectal/corpus_occidental_net.jsonl",
    "sinteticos/genera_corpus_sintetico.py",
]
faltan = [r for r in requeridos if not os.path.exists(os.path.join(PROJECT_DIR, r))]
if faltan:
    raise FileNotFoundError(
        "Faltan estos archivos en " + PROJECT_DIR + \
        " (revisa PROJECT_DIR o sube lo que falte a tu Drive):\n  - " + \
        "\n  - ".join(faltan)
    )
print("Todo encontrado. PROJECT_DIR:", PROJECT_DIR)


Todo encontrado. PROJECT_DIR: /content/drive/MyDrive/AVLizador/scrapeo


## 3. Instalar Ollama

Colab da acceso root dentro de la VM, así que el instalador oficial
funciona directo (sin necesitar `sudo`).


In [11]:
# 1. Instalar zstd
!apt-get install -y zstd

# 2. Instalar ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Verificar
!ollama --version

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 57 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (492 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122579 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Addin

## 4. Arrancar Ollama en segundo plano

Los modelos se guardan en **tu Drive** (`OLLAMA_MODELS`), no en el disco
local de la VM: así, si Colab desconecta la sesión, la próxima vez no hace
falta volver a descargar los ~9-10 GB de qwen2.5:14b.

Nota: leer el modelo desde Drive en cada arranque es algo más lento que
desde disco local — si notas que tarda demasiado en cargar el modelo al
traducir la primera frase de cada sesión, es lo esperable (E/S de Drive),
no un error.


In [19]:
import subprocess, time, requests

# Opción A: modelo en disco local de Colab (se borra al cerrar sesión, ~50GB disponibles)
# No necesita espacio en Drive
OLLAMA_MODELS_DIR = "/root/.ollama/models"
os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)

env = os.environ.copy()
env["OLLAMA_MODELS"] = OLLAMA_MODELS_DIR

log_path = "/content/ollama_serve.log"
log_file = open(log_path, "w")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=subprocess.STDOUT, env=env)

print("Esperando a que Ollama responda...")
listo = False
for _ in range(30):
    try:
        r = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        if r.ok:
            listo = True
            break
    except Exception:
        pass
    time.sleep(2)

if listo:
    print("Ollama activo (PID", ollama_proc.pid, ")")
else:
    print("Ollama no responde todavía. Revisa el log:")
    !cat {log_path}

Esperando a que Ollama responda...
Ollama activo (PID 7752 )


## 5. Descargar qwen2.5:14b (solo la primera vez)

~9-10 GB. Si ya lo descargaste en una sesión anterior y sigue en tu Drive
(`OLLAMA_MODELS_DIR`), esto será casi instantáneo.


In [20]:
!ollama pull qwen2.5:14b

## 6. Comprobar que de verdad está usando la GPU

Fíjate en `size_vram`: si es 0, está en CPU (como en tu portátil) y no
merece la pena seguir — algo ha ido mal con la GPU. Si es mayor que 0
(unos 9-10 GB), va por GPU.


In [21]:
!ollama ps


NAME    ID    SIZE    PROCESSOR    CONTEXT    UNTIL 


## 7. Prueba rápida de segmentación (no llama a Ollama, solo revisa el corpus)


In [22]:
%cd {SINTETICOS_DIR}
!python genera_corpus_sintetico.py --dry-run --verbose


/content/drive/MyDrive/AVLizador/scrapeo/sinteticos
Lèxic carregat: 1604 formes valencianes (palabras_traducidas.json)
2026-09-09 08:30:37,455 [INFO] Ollama: http://localhost:11434
2026-09-09 08:30:38,951 [INFO] Documents llegits: 1778 ({'butlleti': 39, 'biografia': 1, 'glossari': 1394, 'gramatica_normativa': 117, 'nota-de-premsa': 168, 'pagina': 23, 'post': 3, 'legislacio': 3, 'acord-normatiu': 25, 'publicacions': 5})
2026-09-09 08:30:38,952 [INFO] Frases candidates: 48037 → 46315 úniques (1722 duplicades descartades)
2026-09-09 08:30:38,953 [INFO] [--dry-run] 46315 frases seleccionades per a traduir (no es crida cap model).


## 8. Prueba real: 10 frases con GPU

Compara el tiempo con el que viste en local (~30-60 s/frase en CPU). Si la
GPU funciona, debería ser notablemente más rápido.


In [23]:
!python genera_corpus_sintetico.py --limit 10 --verbose


Lèxic carregat: 1604 formes valencianes (palabras_traducidas.json)
2026-09-09 08:30:39,421 [INFO] Ollama: http://localhost:11434
2026-09-09 08:30:40,860 [INFO] Documents llegits: 1778 ({'butlleti': 39, 'biografia': 1, 'glossari': 1394, 'gramatica_normativa': 117, 'nota-de-premsa': 168, 'pagina': 23, 'post': 3, 'legislacio': 3, 'acord-normatiu': 25, 'publicacions': 5})
2026-09-09 08:30:40,861 [INFO] Frases candidates: 48037 → 46315 úniques (1722 duplicades descartades)
2026-09-09 08:30:40,868 [INFO] Frases seleccionades: 10 | ja traduïdes: 0 | pendents: 10
2026-09-09 08:33:53,097 [INFO] [1/10] El nou acadèmic alacantí arriba a l’AVL amb l’objectiu fonam...
2026-09-09 08:33:55,331 [INFO] [2/10] Carles Segura Llopes (Alacant, 1970) és doctor en Filologia ...
2026-09-09 08:33:57,684 [INFO] [3/10] Des de finals de la dècada de 1990 està vinculat a esta univ...
2026-09-09 08:33:59,194 [INFO] [4/10] Ha impartit ensenyaments en diferents titulacions de l’àmbit...
2026-09-09 08:34:03,485 [INFO]

## 9. Revisa tú mismo las traducciones

Antes de lanzar nada más grande, échales un ojo — igual que hicimos en local.


In [24]:
import json

with open(os.path.join(SINTETICOS_DIR, "corpus_sintetic_val_cat.jsonl"), encoding="utf-8") as f:
    for linia in f:
        r = json.loads(linia)
        marca = f"  [SOSPITÓS: {','.join(r['motius_sospita'])}]" if r["motius_sospita"] else ""
        print("VAL:", r["frase_val"])
        print("CAT:", r["frase_cat"], marca)
        print()


VAL: El nou acadèmic alacantí arriba a l’AVL amb l’objectiu fonamental de contribuir a millorar l’ús social del valencià
CAT: El nou acadèmic alacantí arriba a l'AVL amb l'objectiu fonamental de contribuir a millorar l'ús social del valencià 

VAL: Carles Segura Llopes (Alacant, 1970) és doctor en Filologia Catalana per la Universitat d’Alacant.
CAT: Carles Segura Llopes (Alacant, 1970) és doctor en Filologia Catalana per la Universitat d'Alacant. 

VAL: Des de finals de la dècada de 1990 està vinculat a esta universitat, de la qual forma part com a professor titular.
CAT: Des de finals de la dècada de 1990 està vinculat a aquesta universitat, de la qual forma part com a professor titular. 

VAL: Ha impartit ensenyaments en diferents titulacions de l’àmbit humanístic.
CAT: Ha impartit ensenyaments en diferents titulacions d'ambit humanístic. 

VAL: La seua trajectòria acadèmica i professional s’ha desenvolupat al voltant de l’estudi, la descripció i l’ensenyament del valencià, tant sob

## 10. Escalar (cuando estés convencido de la calidad y la velocidad)

Empieza por una muestra, no por el corpus completo de golpe. Cambia
`--sample` al tamaño que quieras probar.

**Importante sobre sesiones de Colab**: la sesión gratuita se desconecta
tras ~90 min de inactividad y tiene un límite de sesión (~12h, y con uso
intenso Google puede limitarte el acceso a GPU una temporada sin avisar
cuánto). El script es reanudable y el archivo de salida vive en tu Drive,
así que si se corta, simplemente:

1. Vuelve a ejecutar las celdas 2, 4, 5 y 6 (montar Drive, arrancar Ollama,
   comprobar el modelo) — no hace falta repetir la instalación (celda 3) si
   sigues en la misma sesión, pero sí si Colab te ha dado una VM nueva.
2. Vuelve a ejecutar la celda de generación (con el mismo `--sample`, o sin
   argumentos para el corpus completo) — retomará justo donde lo dejó, no
   repite ninguna frase ya traducida.


In [ ]:
!python genera_corpus_sintetico.py --sample 5000


Lèxic carregat: 1604 formes valencianes (palabras_traducidas.json)
2026-09-09 08:40:22,367 [INFO] Ollama: http://localhost:11434
2026-09-09 08:40:23,762 [INFO] Documents llegits: 1778 ({'butlleti': 39, 'biografia': 1, 'glossari': 1394, 'gramatica_normativa': 117, 'nota-de-premsa': 168, 'pagina': 23, 'post': 3, 'legislacio': 3, 'acord-normatiu': 25, 'publicacions': 5})
2026-09-09 08:40:23,762 [INFO] Frases candidates: 48037 → 46315 úniques (1722 duplicades descartades)
2026-09-09 08:40:23,788 [INFO] Frases seleccionades: 5000 | ja traduïdes: 0 | pendents: 5000
2026-09-09 08:41:58,068 [INFO] [1/5000] En una societat pobra, en què alimentar-se abundantment era ...
2026-09-09 08:42:01,064 [INFO] [2/5000] Els rectors de les universitats rebran, a més, el tractament...
2026-09-09 08:42:02,413 [INFO] [3/5000] La paraula besamans, també usada en singular,...
2026-09-09 08:42:04,808 [INFO] [4/5000] o en un polígon industrial, per exemple, s’escriuen amb núme...
2026-09-09 08:42:07,082 [INFO] [5

## 11. Exportar solo los pares limpios (si quieres repetirlo tras revisar más)


In [ ]:
!python genera_corpus_sintetico.py --nomes-exporta
